# Pretraining and Fine-Tuning of LLMs

## Learning Objectives
- Understand pretraining approaches using large unlabelled datasets.
- Learn how fine-tuning adapts models to downstream tasks.

## Introduction
Pretraining captures general language understanding; fine-tuning customizes models for specific applications.

## Core Concepts
- **Self-Supervised Learning:** Predict masked tokens or next tokens.
- **Fine-Tuning:** Training on task-specific labeled data.
- **Transfer Learning:** Leveraging pretrained knowledge.

## Example
Fine-tuning BERT on a classification task.

In [ ]:
import torch
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

# 1. Load the dataset
print("Loading the IMDb dataset...")
dataset = load_dataset('imdb')

# 2. Load the tokenizer and model
print("Loading the pre-trained BERT model and tokenizer...")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# 3. Preprocess the data
def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True)

print("Tokenizing the dataset...")
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Use a smaller subset of the data for faster training (optional)
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(1000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(1000))

# 4. Set up training arguments
training_args = TrainingArguments(
    output_dir='./results',          # output directory
    num_train_epochs=3,              # total number of training epochs
    per_device_train_batch_size=8,   # batch size per device during training
    per_device_eval_batch_size=8,    # batch size for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for storing logs
    logging_steps=10,
    evaluation_strategy="epoch",     # evaluate each epoch
)

# 5. Create the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
)

# 6. Fine-tune the model
print("Starting the fine-tuning process...")
trainer.train()

# 7. Evaluate the model
print("Evaluating the model...")
eval_results = trainer.evaluate()

print(f"Evaluation results: {eval_results}")

# 8. Make a prediction
print("Making a prediction on a sample text...")
text = "This is a fantastic movie! I really enjoyed it."
inputs = tokenizer(text, return_tensors="pt")
outputs = model(**inputs)
logits = outputs.logits
predicted_class = torch.argmax(logits, dim=1).item()

print(f"Text: '{text}'")
print(f"Predicted class: {'Positive' if predicted_class == 1 else 'Negative'}")

Loading dataset...
Tokenizing data...


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

/Users/johnmoses/miniforge3/envs/mforge312/lib/python3.12/site-packages/datasets/arrow_dataset.py:403: FutureWarning: The output of `to_tf_dataset` will change when a passing single element list for `labels` or `columns` in the next datasets version. To return a tuple structure rather than dict, pass a single string.
Old behaviour: columns=['a'], labels=['labels'] -> (tf.Tensor, tf.Tensor)  
             : columns='a', labels='labels' -> (tf.Tensor, tf.Tensor)  
New behaviour: columns=['a'],labels=['labels'] -> ({'a': tf.Tensor}, {'labels': tf.Tensor})  
             : columns='a', labels='labels' -> (tf.Tensor, tf.Tensor) 
  warnings.warn(
2025-09-02 17:49:43.609320: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Pro
2025-09-02 17:49:43.609351: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2025-09-02 17:49:43.609354: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 10.67 GB
I0000 00:00:1756831783.609367 1024737 pluggab

Loading pre-trained BERT model...


TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


TypeError: 'builtins.safe_open' object is not iterable

## Exercise
Try fine-tuning on a custom text classification or sentiment dataset.

In [ ]:
# Your code here

## Summary
- Pretraining and fine-tuning form the backbone of LLM adaptation.
- Fine-tuning allows specialized high-performance applications.


## Further Reading
- [BERT Fine-Tuning Guide](https://huggingface.co/transformers/training.html)
- [Transfer Learning in NLP](https://ruder.io/nlp-transfer-learning/)
